# apertus-eval-prep — ranking stability on Colab

Runtime → Change runtime type → **T4 GPU**.

Do **not** Run all. One session = setup cells + **one** sweep cell + the save at the bottom of that cell.

Free Colab dies after 1–2 hours. Each sweep cell is one model × one factor (usually **one** 800-item run). After it prints `[800/800]`, results are copied to **Google Drive** and a zip downloads to your Mac.

Next day: Run setup again (Drive restore), then the **next** sweep cell. Finished `config_hash` rows skip.

Use `results/registry_paper.jsonl` (not the n=4 smoke `registry.jsonl`).

In [ ]:
import os
if os.path.exists("pyproject.toml") and os.path.exists("data/eval_set.jsonl"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[gpu,viz]"
!git log -1 --oneline

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0))
if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")

## Persist to Google Drive

Authorize Drive when prompted. Files live in `MyDrive/apertus-eval-prep-paper/` so a runtime reset does not wipe finished cells.

Also download the zip to your Mac as a second copy.

In [ ]:
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").glob("*.json")):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")

SWEEP = [
    "python", "-m", "apertus_eval_prep", "sweep",
    "--config", "configs/experiments/stability.yaml",
    "--profile", "t4",
    "--out-dir", "results/runs",
    "--registry", "results/registry_paper.jsonl",
]

def save_paper():
    import subprocess
    DRIVE.mkdir(parents=True, exist_ok=True)
    (DRIVE / "runs").mkdir(exist_ok=True)
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    import subprocess
    cmd = SWEEP + list(extra)
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --out-dir results/runs --registry results/registry_paper.jsonl | head -n 50

## Session cells (run one per Colab window)

Each cell below is ~one 800-item job except `rest` cells (remaining factors for that model; skip if you only have time for control).

Suggested 3-day plan:

1. SmolLM2 control (already on GitHub — this cell should mostly `skip`) then Qwen-3B **control**
2. Phi-3.5 **control** then Qwen-3B rest *or* stop after control if quota is gone
3. Remaining factors / 7B int4

Wait until `[800/800]` and the Drive zip download finish before closing the tab.

In [ ]:
# Day 1a — SmolLM2 control (skip if already in registry)
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "control")

In [ ]:
# Day 1b — Qwen 3B control
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "control")

In [ ]:
# Day 2a — Phi-3.5 control (native transformers; do not use Hub modeling_phi3.py)
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "control")

In [ ]:
# Later — remaining SmolLM2 factors (control is skipped)
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct")

In [ ]:
# Later — remaining Qwen 3B factors
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct")

In [ ]:
# Later — remaining Phi-3.5 factors
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct")

In [ ]:
# T4 only keeps 7B int4 (fp16 / int8 / vLLM are skipped)
sweep("--only-model", "Qwen/Qwen2.5-7B-Instruct")

## Report (only after several cells exist)

cwd must be `/content/apertus-eval-prep`. Do not run this instead of a sweep.

In [ ]:
from google.colab import files
from pathlib import Path

assert Path("results/registry_paper.jsonl").exists(), "No paper registry in this runtime. Restore from Drive first."
!python -m apertus_eval_prep report --registry results/registry_paper.jsonl --out reports/stability_paper
!python -m apertus_eval_prep paper-tables --registry results/registry_paper.jsonl --out paper/_generated_tables.md
!zip -r paper_matrix_artifacts.zip results/runs results/registry_paper.jsonl reports/stability_paper paper/_generated_tables.md
print("zip bytes", Path("paper_matrix_artifacts.zip").stat().st_size)
files.download("paper_matrix_artifacts.zip")

Unpack the zip (or Drive folder) into the Mac clone. Commit `results/registry_paper.jsonl` and new `results/runs/*.json`. Do not edit numbers.